# 🎧 Desafio CP05 — Meu Analisador Léxico de Mercado
## Comandos de Playlist (Spotify)

**Integrantes:**

*   Diego Azevedo Dias Ignacio - 2392072
*   Gabriella Pereira Rodrigues - 2334425
*   Giovanna Bispo Da Silva - 2367880
*   José Henrique Guimarães Galvão Silva - 2499948





**Exemplo de entrada do domínio:**

```
PLAYLIST "Treino" ADICIONAR "Nome da faixa" DURACAO 03:45 GENERO rock ORDENAR popularidade
```

Este notebook implementa um **analisador léxico** com [Lark](https://lark-parser.readthedocs.io/)
para uma mini-linguagem de comandos de playlist, com:
tabela de tokens documentada, 23 tipos de token, 13 palavras reservadas (`/i` + `\b`),
literais por regex, resolução de conflitos de prioridade, comentários `#` ignorados,
erros léxicos com linha/coluna/dicas, interface `ipywidgets` e aba de pós-processamento (bônus).

##Preparando o Ambiente


In [ ]:
# Instalação (Google Colab / ambiente novo)
# !pip install -q lark ipywidgets

from lark import Lark, Token
from lark.exceptions import UnexpectedCharacters
import ipywidgets as widgets
from IPython.display import display, HTML
import html as _html
import re
from datetime import datetime

print("Ambiente pronto ✅")

Ambiente pronto ✅


## 1) GRAMÁTICA DO LEXER



In [ ]:
# ============================================================
# GRAMÁTICA DO LEXER — Tema 5: Comandos de Playlist (Spotify)
# ============================================================
gramatica_desafio = r"""
start: _token*

_token: PLAYLIST | CRIAR | ADICIONAR | REMOVER | TOCAR | ORDENAR | EMBARALHAR
      | DURACAO | GENERO | VOLUME | LIMITE | POR | COMPARTILHAR
      | URI_SPOTIFY | EMAIL | DATA | TEMPO | PERCENTUAL | MULTIPLICADOR
      | TEXTO | NUMERO | IDENT | VIRGULA

// ---------- 1) Tokens especiais do domínio (prioridade mais alta) ----------
// CONFLITO 1 (resolvido): "spotify:track:XXXX" começaria como IDENT("spotify") + ':' inválido.
// Prioridade 9 faz o lexer casar a URI inteira ANTES de tentar IDENT. É o caso "pix@loja.com.br" do enunciado.
URI_SPOTIFY.9: /spotify:(?:track|album|artist|playlist):[A-Za-z0-9]{22}\b/i

// CONFLITO 2 (resolvido): e-mail x IDENT + '@'. Prioridade 8 > IDENT(1) garante o lexema inteiro.
EMAIL.8:       /[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(?:\.[A-Za-z]{2,})+/

// ---------- 2) Literais (mín. 3 tipos com regex) ----------
// CONFLITO 3 (resolvido): DATA/TEMPO x NUMERO. Sem prioridade, "03:45" viraria NUMERO(03) + erro em ':'.
DATA.7:        /\d{2}\/\d{2}\/\d{4}/
TEMPO.7:       /\d{1,2}:[0-5]\d/
PERCENTUAL.6:  /\d{1,3}%/
MULTIPLICADOR.6: /\d+x\b/i

// ---------- 3) Palavras reservadas (13, todas /i + \b) ----------
// CONFLITO 4 (resolvido): palavra reservada x IDENT. Prioridade 5 > 1; o \b evita casar "playlistao".
PLAYLIST.5:     /playlist\b/i
CRIAR.5:        /criar\b/i
ADICIONAR.5:    /adicionar\b/i
REMOVER.5:      /remover\b/i
TOCAR.5:        /tocar\b/i
ORDENAR.5:      /ordenar\b/i
EMBARALHAR.5:   /embaralhar\b/i
DURACAO.5:      /dura[cç][aã]o\b/i
GENERO.5:       /g[eê]nero\b/i
VOLUME.5:       /volume\b/i
LIMITE.5:       /limite\b/i
POR.5:          /por\b/i
COMPARTILHAR.5: /compartilhar\b/i

// ---------- 4) Literais genéricos e identificadores ----------
TEXTO.4:   /"[^"\n]*"/
NUMERO.2:  /\d+/
IDENT.1:   /[A-Za-zÀ-ÿ_][A-Za-zÀ-ÿ0-9_\-]*/
VIRGULA.1: /,/

// ---------- 5) Ruído (ignorado) ----------
COMENTARIO: /#[^\n]*/
%ignore COMENTARIO
%ignore /[ \t\r\n]+/
"""

lexer_desafio = Lark(gramatica_desafio, parser="lalr", lexer="basic")

# ---------- Dicas amigáveis de erro, específicas do domínio ----------
DICAS_DESAFIO = {
    '"': 'Aspas abertas e não fechadas — nomes de playlist/faixa precisam fechar: PLAYLIST "Treino".',
    ':': 'Duração inválida. Use mm:ss com segundos entre 00 e 59 (ex.: DURACAO 03:45) ou uma URI spotify:track:<22 caracteres>.',
    '@': 'E-mail malformado em COMPARTILHAR. Use algo como COMPARTILHAR nome@empresa.com.br.',
    '%': 'Porcentagem inválida. Escreva o número colado ao %, com 1 a 3 dígitos (ex.: VOLUME 80%).',
    '/': 'Data inválida. Use o formato dd/mm/aaaa (ex.: 20/07/2026).',
    '$': 'Não use símbolos monetários aqui: esta linguagem controla playlists, não pagamentos.',
    '&': 'Use VIRGULA (,) para separar itens de uma lista de faixas, não "&".',
    "'": 'Use aspas duplas (") para nomes de playlist e faixas.',
}

def tokenizar_desafio(texto):
    try:
        return list(lexer_desafio.lex(texto))
    except UnexpectedCharacters as erro:
        erro.dica = DICAS_DESAFIO.get(erro.char, "Caractere fora do alfabeto da linguagem de playlist.")
        raise

print("Lexer construído ✅ —", len(lexer_desafio.terminals), "terminais")

Lexer construído ✅ — 25 terminais


## 2) Funções de apresentação (texto colorido, tabela e erro léxico)

In [ ]:
CORES = {
    "PLAYLIST": "#1DB954", "CRIAR": "#1DB954", "ADICIONAR": "#1DB954", "REMOVER": "#1DB954",
    "TOCAR": "#1DB954", "ORDENAR": "#1DB954", "EMBARALHAR": "#1DB954", "DURACAO": "#1DB954",
    "GENERO": "#1DB954", "VOLUME": "#1DB954", "LIMITE": "#1DB954", "POR": "#1DB954",
    "COMPARTILHAR": "#1DB954",
    "TEXTO": "#c2410c", "NUMERO": "#1d4ed8", "TEMPO": "#7c3aed", "DATA": "#7c3aed",
    "PERCENTUAL": "#0891b2", "MULTIPLICADOR": "#0891b2",
    "URI_SPOTIFY": "#be185d", "EMAIL": "#be185d",
    "IDENT": "#374151", "VIRGULA": "#9ca3af",
}

def _cor(tipo):
    return CORES.get(tipo, "#374151")

def texto_colorido_html(texto, tokens):
    partes, pos = [], 0
    for t in tokens:
        ini, fim = t.start_pos, t.end_pos
        partes.append('<span style="color:#9ca3af">' + _html.escape(texto[pos:ini]) + "</span>")
        partes.append(f'<span style="color:{_cor(t.type)};font-weight:600" title="{t.type}">'
                      + _html.escape(texto[ini:fim]) + "</span>")
        pos = fim
    partes.append('<span style="color:#9ca3af">' + _html.escape(texto[pos:]) + "</span>")
    return ('<pre style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;'
            'padding:12px;white-space:pre-wrap;font-family:Consolas,monospace">'
            + "".join(partes) + "</pre>")

def tabela_tokens_html(tokens):
    linhas = "".join(
        f'<tr><td style="padding:4px 10px">{i}</td>'
        f'<td style="padding:4px 10px;color:{_cor(t.type)};font-weight:600">{t.type}</td>'
        f'<td style="padding:4px 10px"><code>{_html.escape(str(t))}</code></td>'
        f'<td style="padding:4px 10px">{t.line}</td>'
        f'<td style="padding:4px 10px">{t.column}</td></tr>'
        for i, t in enumerate(tokens, 1))
    return (f'<p><b>{len(tokens)} tokens reconhecidos</b></p>'
            '<table style="border-collapse:collapse;font-family:Segoe UI,sans-serif;font-size:13px">'
            '<tr style="background:#111;color:#fff"><th style="padding:6px 10px">#</th>'
            '<th style="padding:6px 10px">Token</th><th style="padding:6px 10px">Lexema</th>'
            '<th style="padding:6px 10px">Linha</th><th style="padding:6px 10px">Coluna</th></tr>'
            + linhas + "</table>")

def erro_lexico_html(texto, erro):
    linhas = texto.splitlines() or [""]
    linha_txt = linhas[erro.line - 1] if erro.line - 1 < len(linhas) else ""
    seta = " " * (erro.column - 1) + "^"
    dica = getattr(erro, "dica", "Caractere fora do alfabeto da linguagem de playlist.")
    return ('<div style="background:#fef2f2;border-left:5px solid #dc2626;border-radius:8px;padding:12px;'
            'font-family:Segoe UI,sans-serif">'
            f'<b style="color:#b91c1c">❌ Erro léxico na linha {erro.line}, coluna {erro.column}</b>'
            f'<p>Caractere inesperado: <code>{_html.escape(repr(erro.char))}</code></p>'
            f'<pre style="background:#fff;padding:8px;border-radius:6px">{_html.escape(linha_txt)}\n{seta}</pre>'
            f'<p>💡 <b>Dica:</b> {_html.escape(dica)}</p>'
            '<p>💡 <b>Dica:</b> comandos válidos começam por CRIAR, PLAYLIST, ADICIONAR, REMOVER, TOCAR, '
            'ORDENAR, EMBARALHAR, VOLUME, LIMITE, DURACAO, GENERO, POR ou COMPARTILHAR.</p></div>')

print("Renderizadores prontos ✅")

Renderizadores prontos ✅


## 3) Pós-processamento (bônus): conversões, mascaramento LGPD, resumo e alertas

In [ ]:
def tempo_para_segundos(lexema):
    m, s = lexema.split(":")
    return int(m) * 60 + int(s)

def mascarar_email(lexema):
    """LGPD: preserva apenas a inicial do usuário e o domínio."""
    usuario, dominio = lexema.split("@", 1)
    return usuario[0] + "*" * max(len(usuario) - 1, 1) + "@" + dominio

def mascarar_uri(lexema):
    prefixo, tipo, ident = lexema.split(":")
    return f"{prefixo}:{tipo}:{ident[:4]}{'*' * 14}{ident[-4:]}"

def converter(token):
    """Converte o lexema (texto) no valor semântico do domínio."""
    t, v = token.type, str(token)
    if t == "NUMERO":        return int(v)
    if t == "TEMPO":         return tempo_para_segundos(v)
    if t == "PERCENTUAL":    return int(v[:-1])
    if t == "MULTIPLICADOR": return int(v[:-1])
    if t == "DATA":
        try:    return datetime.strptime(v, "%d/%m/%Y").date()
        except ValueError: return None
    if t == "TEXTO":         return v[1:-1]
    if t == "EMAIL":         return mascarar_email(v)
    if t == "URI_SPOTIFY":   return mascarar_uri(v)
    return v

def resumo_html(tokens):
    contagem = {}
    for t in tokens:
        contagem[t.type] = contagem.get(t.type, 0) + 1

    duracoes = [tempo_para_segundos(str(t)) for t in tokens if t.type == "TEMPO"]
    faixas   = sum(1 for t in tokens if t.type in ("TEXTO", "URI_SPOTIFY")) - contagem.get("PLAYLIST", 0)
    total_s  = sum(duracoes)
    reservadas = sum(n for k, n in contagem.items() if k in {
        "PLAYLIST","CRIAR","ADICIONAR","REMOVER","TOCAR","ORDENAR","EMBARALHAR",
        "DURACAO","GENERO","VOLUME","LIMITE","POR","COMPARTILHAR"})

    alertas = []
    for t in tokens:
        if t.type == "PERCENTUAL" and converter(t) > 100:
            alertas.append(f"🔊 Volume {t} acima de 100% (linha {t.line}).")
        if t.type == "TEMPO" and tempo_para_segundos(str(t)) < 30:
            alertas.append(f"⏱️ Faixa com duração muito curta ({t}) na linha {t.line}.")
        if t.type == "TEXTO" and len(str(t)) == 2:
            alertas.append(f"📝 Nome vazio entre aspas na linha {t.line}.")
        if t.type == "EMAIL":
            alertas.append(f"🔒 LGPD: e-mail detectado na linha {t.line} — mascarado como {converter(t)}.")
    if not any(t.type in ("PLAYLIST", "CRIAR") for t in tokens):
        alertas.append("⚠️ Nenhum comando PLAYLIST/CRIAR encontrado — a playlist alvo não foi identificada.")
    if not alertas:
        alertas.append("✅ Nenhum alerta: comando íntegro.")

    tot = ("".join(f'<tr><td style="padding:3px 10px;color:{_cor(k)};font-weight:600">{k}</td>'
                   f'<td style="padding:3px 10px">{v}</td></tr>'
                   for k, v in sorted(contagem.items(), key=lambda x: (-x[1], x[0]))))

    conv = "".join(
        f'<tr><td style="padding:3px 10px">{t.type}</td>'
        f'<td style="padding:3px 10px"><code>{_html.escape(str(t))}</code></td>'
        f'<td style="padding:3px 10px"><b>{_html.escape(str(converter(t)))}</b></td></tr>'
        for t in tokens if t.type in ("NUMERO","TEMPO","DATA","PERCENTUAL","MULTIPLICADOR","EMAIL","URI_SPOTIFY"))

    return f"""
    <div style="font-family:Segoe UI,sans-serif;font-size:13px">
      <h4>📊 Resumo da comanda de playlist</h4>
      <ul>
        <li>Total de tokens: <b>{len(tokens)}</b></li>
        <li>Palavras reservadas usadas: <b>{reservadas}</b></li>
        <li>Itens/faixas referenciados: <b>{max(faixas, 0)}</b></li>
        <li>Tempo total declarado: <b>{total_s // 60:02d}:{total_s % 60:02d}</b> ({total_s} s)</li>
      </ul>
      <h4>🔢 Totais por tipo de token</h4>
      <table style="border-collapse:collapse"><tr style="background:#111;color:#fff">
        <th style="padding:5px 10px">Token</th><th style="padding:5px 10px">Qtde</th></tr>{tot}</table>
      <h4>🔄 Conversão de lexemas em valores (+ mascaramento LGPD)</h4>
      <table style="border-collapse:collapse"><tr style="background:#111;color:#fff">
        <th style="padding:5px 10px">Token</th><th style="padding:5px 10px">Lexema</th>
        <th style="padding:5px 10px">Valor convertido</th></tr>{conv or '<tr><td colspan=3 style="padding:5px 10px">—</td></tr>'}</table>
      <h4>🚨 Alertas</h4>
      <ul>{''.join(f'<li>{_html.escape(a)}</li>' for a in alertas)}</ul>
    </div>"""

print("Pós-processamento pronto ✅")

Pós-processamento pronto ✅


## 4) Casos de teste (3 válidos + 3 inválidos)

In [ ]:
casos_desafio = {
    "✅ Válido 1 — comando básico":
        'PLAYLIST "Treino" ADICIONAR "Nome da faixa" DURACAO 03:45 GENERO rock ORDENAR popularidade',

    "✅ Válido 2 — criação + URI + comentários":
        '# monta a playlist de foco e limita o tamanho\n'
        'CRIAR PLAYLIST "Foco Total" LIMITE 25 VOLUME 80% EMBARALHAR\n'
        'ADICIONAR 3x spotify:track:4cOdK2wGLETKBW3PvgPWqT POR "Rick Astley"  # faixa fixa',

    "✅ Válido 3 — datas, acentos e compartilhamento":
        'playlist "Festa da Firma" remover "Faixa antiga" duração 12:05 gênero MPB\n'
        'TOCAR EM 20/07/2026 COMPARTILHAR diego.inacio@empresa.com.br, ana@empresa.com',

    "❌ Inválido 1 — aspas não fechadas":
        'PLAYLIST "Treino ADICIONAR "Faixa"',

    "❌ Inválido 2 — duração fora do padrão mm:ss":
        'PLAYLIST "Treino" DURACAO 03:75 GENERO rock',

    "❌ Inválido 3 — símbolo fora do alfabeto":
        'VOLUME 80 $ ORDENAR popularidade',
}

def testar(tokenizar, casos):
    """Roda todos os casos e mostra um relatório rápido no console."""
    for nome, texto in casos.items():
        try:
            toks = tokenizar(texto)
            print(f"{nome:<45} -> {len(toks):>3} tokens: {[t.type for t in toks][:8]}...")
        except UnexpectedCharacters as e:
            print(f"{nome:<45} -> ERRO na linha {e.line}, coluna {e.column} ({e.char!r}) | {e.dica}")

testar(tokenizar_desafio, casos_desafio)

✅ Válido 1 — comando básico                   ->  10 tokens: ['PLAYLIST', 'TEXTO', 'ADICIONAR', 'TEXTO', 'DURACAO', 'TEMPO', 'GENERO', 'IDENT']...
✅ Válido 2 — criação + URI + comentários      ->  13 tokens: ['CRIAR', 'PLAYLIST', 'TEXTO', 'LIMITE', 'NUMERO', 'VOLUME', 'PERCENTUAL', 'EMBARALHAR']...
✅ Válido 3 — datas, acentos e compartilhamento ->  15 tokens: ['PLAYLIST', 'TEXTO', 'REMOVER', 'TEXTO', 'DURACAO', 'TEMPO', 'GENERO', 'IDENT']...
❌ Inválido 1 — aspas não fechadas             -> ERRO na linha 1, coluna 34 ('"') | Aspas abertas e não fechadas — nomes de playlist/faixa precisam fechar: PLAYLIST "Treino".
❌ Inválido 2 — duração fora do padrão mm:ss   -> ERRO na linha 1, coluna 29 (':') | Duração inválida. Use mm:ss com segundos entre 00 e 59 (ex.: DURACAO 03:45) ou uma URI spotify:track:<22 caracteres>.
❌ Inválido 3 — símbolo fora do alfabeto       -> ERRO na linha 1, coluna 11 ('$') | Não use símbolos monetários aqui: esta linguagem controla playlists, não pagamentos.


## 5) Interface com ipywidgets (com aba de pós-processamento — bônus)

In [ ]:
def interface_lexer(titulo, tokenizar, casos):
    """Interface genérica reutilizável: serve para QUALQUER lexer seu."""
    seletor = widgets.Dropdown(options=list(casos), description="Casos:",
                               layout=widgets.Layout(width="60%"))
    entrada = widgets.Textarea(value=list(casos.values())[0],
                               layout=widgets.Layout(width="95%", height="150px"))
    botao = widgets.Button(description="▶ Analisar", button_style="success")
    saida_analise = widgets.Output()
    saida_resumo = widgets.Output()
    abas = widgets.Tab(children=[saida_analise, saida_resumo])
    abas.set_title(0, "🔎 Análise léxica")
    abas.set_title(1, "📊 Pós-processamento")

    def analisar(_):
        saida_analise.clear_output()
        saida_resumo.clear_output()
        try:
            toks = tokenizar(entrada.value)
            with saida_analise:
                display(HTML(texto_colorido_html(entrada.value, toks)))
                display(HTML(tabela_tokens_html(toks)))
            with saida_resumo:
                display(HTML(resumo_html(toks)))
        except UnexpectedCharacters as e:
            with saida_analise:
                display(HTML(erro_lexico_html(entrada.value, e)))
            with saida_resumo:
                display(HTML('<p style="font-family:Segoe UI">Sem resumo: corrija o erro léxico primeiro.</p>'))

    def escolher(m):
        entrada.value = casos[m["new"]]
        analisar(None)

    seletor.observe(escolher, names="value")
    botao.on_click(analisar)
    display(widgets.HTML(f"<h3>{titulo}</h3>"), seletor, entrada, botao, abas)
    analisar(None)

interface_lexer("🎧 Meu Analisador Léxico — Comandos de Playlist (Spotify)",
                tokenizar_desafio, casos_desafio)

HTML(value='<h3>🎧 Meu Analisador Léxico — Comandos de Playlist (Spotify)</h3>')

Dropdown(description='Casos:', layout=Layout(width='60%'), options=('✅ Válido 1 — comando básico', '✅ Válido 2…

Textarea(value='PLAYLIST "Treino" ADICIONAR "Nome da faixa" DURACAO 03:45 GENERO rock ORDENAR popularidade', l…

Button(button_style='success', description='▶ Analisar', style=ButtonStyle())

## 6) Diário de ambiguidade

O conflito mais interessante apareceu exatamente no estilo do caso `pix@loja.com.br` do enunciado, mas na
versão Spotify: o lexema **`spotify:track:4cOdK2wGLETKBW3PvgPWqT`**. Na primeira versão da gramática eu tinha
só `IDENT`, `NUMERO` e `TEMPO`, e o lexer quebrava a URI em `IDENT("spotify")` e depois estourava
`UnexpectedCharacters` no `:` — ou, pior, casava `TEMPO` em pedaços que pareciam `mm:ss`. O `:` é ambíguo na
minha linguagem porque serve para **duas coisas diferentes**: separar minutos de segundos em `DURACAO 03:45`
e separar os campos de uma URI do Spotify. Resolvi criando o terminal `URI_SPOTIFY` com **prioridade 9** —
a maior de todas — e uma regex bem específica (`spotify:(?:track|album|artist|playlist):[A-Za-z0-9]{22}\b`),
que exige o prefixo literal, um tipo conhecido e exatamente 22 caracteres base62 do ID. Como o Lark resolve
empates de posição pela prioridade do terminal, a URI inteira é consumida como **um único token** antes que
`IDENT` ou `TEMPO` tenham chance de agir; e como a regex é restritiva, ela nunca "rouba" um `DURACAO 03:45`
legítimo. O mesmo raciocínio foi aplicado ao `EMAIL` (prioridade 8, para não virar `IDENT` + `@` inválido) e
ao par `DATA`/`TEMPO` (prioridade 7 sobre `NUMERO`, que tem prioridade 2 — senão `03:45` viraria `NUMERO(03)`
seguido de erro). Por fim, as 13 palavras reservadas ficaram com prioridade 5 contra o `IDENT` de prioridade 1,
sempre com `\b`, para que `playlistao` continue sendo um identificador comum e não `PLAYLIST` + `ao`.

### Resumo das prioridades (do mais forte para o mais fraco)

| Prioridade | Terminais | Por quê |
|---|---|---|
| 9 | `URI_SPOTIFY` | vence `IDENT` + `:` e o falso `TEMPO` dentro da URI |
| 8 | `EMAIL` | vence `IDENT` + `@` |
| 7 | `DATA`, `TEMPO` | vencem `NUMERO` (senão `03:45` → `03` + erro) |
| 6 | `PERCENTUAL`, `MULTIPLICADOR` | vencem `NUMERO` (`80%`, `3x` são unidades do domínio) |
| 5 | 13 palavras reservadas | vencem `IDENT`; `\b` evita prefixos de palavras maiores |
| 4 | `TEXTO` | literal delimitado, não colide com o resto |
| 2 | `NUMERO` | genérico, só depois dos numéricos especializados |
| 1 | `IDENT`, `VIRGULA` | fallback mais genérico da linguagem |